In [16]:
import pandas as pd
import fastf1 as f1

Working on a sample data for a session **(2023 Austrian Grand Prix)** to point out what necessary functions will be required for data cleaning process.

#### Laps Data

In [2]:
df_loc = "f1_parquet_data/2023/Round_9_Austrian_Grand_Prix/R"
sample_df_laps = pd.read_parquet(f"{df_loc}/laps.parquet")
sample_df_laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:03:05.095000,VER,1,0 days 00:01:17.639000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:31.613000,...,True,Red Bull Racing,0 days 01:01:47.219000,2023-07-02 13:02:48.211,124,1.0,False,,False,False
1,0 days 01:05:00.574000,VER,1,0 days 00:01:55.479000,2.0,1.0,NaT,0 days 01:04:57.200000,0 days 00:00:31.698000,0 days 00:00:46.293000,...,True,Red Bull Racing,0 days 01:03:05.095000,2023-07-02 13:04:06.087,4,1.0,False,,False,False
2,0 days 01:07:05.295000,VER,1,0 days 00:02:04.721000,3.0,2.0,0 days 01:05:13.560000,NaT,0 days 00:00:37.877000,0 days 00:00:46.298000,...,False,Red Bull Racing,0 days 01:05:00.574000,2023-07-02 13:06:01.566,41,1.0,False,,False,False
3,0 days 01:08:14.986000,VER,1,0 days 00:01:09.691000,4.0,2.0,NaT,NaT,0 days 00:00:17.618000,0 days 00:00:30.970000,...,False,Red Bull Racing,0 days 01:07:05.295000,2023-07-02 13:08:06.287,1,1.0,False,,False,True
4,0 days 01:09:25.012000,VER,1,0 days 00:01:10.026000,5.0,2.0,NaT,NaT,0 days 00:00:17.716000,0 days 00:00:31.158000,...,False,Red Bull Racing,0 days 01:08:14.986000,2023-07-02 13:09:15.978,1,1.0,False,,False,True


In [3]:
print(sample_df_laps.columns)
print("Shape of the dataset: ",sample_df_laps.shape)

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')
Shape of the dataset:  (1354, 31)


There are in total 31 columns in the dataset. The first step that is required is to remove the unnecessary columns from the dataset. 

In [4]:
to_drop = ["DriverNumber","Sector1SessionTime","Sector2SessionTime","Sector3SessionTime", "SpeedI1","SpeedI2","SpeedFL","SpeedST","LapStartTime","LapStartDate","FastF1Generated"]
clean_df_laps = sample_df_laps.drop(to_drop, axis=1)
clean_df_laps.shape

(1354, 20)

In [5]:
sample_df_res = pd.read_parquet(f"{df_loc}/results.parquet")
sample_df_res.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:25:33.607000,Finished,26.0,71.0
16,16,C LECLERC,LEC,leclerc,Ferrari,F91536,ferrari,Charles,Leclerc,Charles Leclerc,...,2.0,2,2.0,NaT,NaT,NaT,0 days 00:00:05.155000,Finished,18.0,71.0
11,11,S PEREZ,PER,perez,Red Bull Racing,3671C6,red_bull,Sergio,Perez,Sergio Perez,...,3.0,3,15.0,NaT,NaT,NaT,0 days 00:00:17.188000,Finished,15.0,71.0
4,4,L NORRIS,NOR,norris,McLaren,F58020,mclaren,Lando,Norris,Lando Norris,...,4.0,4,4.0,NaT,NaT,NaT,0 days 00:00:26.327000,Finished,12.0,71.0
14,14,F ALONSO,ALO,alonso,Aston Martin,358C75,aston_martin,Fernando,Alonso,Fernando Alonso,...,5.0,5,7.0,NaT,NaT,NaT,0 days 00:00:30.317000,Finished,10.0,71.0


In [6]:
clean_df_laps["Driver"].unique()

array(['VER', 'GAS', 'PER', 'ALO', 'LEC', 'STR', 'SAR', 'MAG', 'DEV',
       'TSU', 'ALB', 'ZHO', 'HUL', 'OCO', 'NOR', 'HAM', 'SAI', 'RUS',
       'BOT', 'PIA'], dtype=object)

In [7]:
# Creating the driver mapping
driver_map = dict(zip(sample_df_res['Abbreviation'], sample_df_res['FirstName'] + ' ' + sample_df_res['LastName']))

# Replace Driver abbreviation with their full Name
clean_df_laps['Driver'] = clean_df_laps['Driver'].map(driver_map).fillna(clean_df_laps['Driver'])

# Encoding the Compound Tyres
compound_map = {"SOFT": 1, "MEDIUM": 2, "HARD": 3, "INTERMEDIATE": 4, "WET": 5}
clean_df_laps["Compound"] = clean_df_laps["Compound"].map(compound_map).fillna(clean_df_laps['Compound'])


In [8]:
# Converting the LapTime, PitInTime, PitOutTime, Sector1Time, Sector2Time, Sector3Time to seconds.
col_to_sec = ["LapTime", "PitOutTime", "PitInTime", "Sector1Time", "Sector2Time", "Sector3Time"]
for t in col_to_sec:
    clean_df_laps[t] = clean_df_laps[t].dt.total_seconds()


**Track Status**<br>
1 - Track Clear / Normal Conditions<br>
2 - Yellow Flag<br>
4 - Safety Car Deployed<br>
5 - Red Flag<br>
6 - Virtual Safety Car Deployed<br>
7 - Virtual Safety Car Ending<br>

In [9]:
clean_df_laps['TrackStatus'].unique()
print("Laps shape before removing non-clean laps", clean_df_laps.shape)
clean_df_laps = clean_df_laps[clean_df_laps["TrackStatus"] == "1"]
print("Laps shape after removing non clean laps", clean_df_laps.shape)

clean_df_laps['TrackStatus'].unique()

Laps shape before removing non-clean laps (1354, 20)
Laps shape after removing non clean laps (1225, 20)


array(['1'], dtype=object)

In [10]:
# Remove the pitin and pitout laps
clean_df_laps = clean_df_laps[clean_df_laps["PitInTime"].isna()]
clean_df_laps = clean_df_laps[clean_df_laps["PitOutTime"].isna()]
clean_df_laps.shape

(1168, 20)

In [11]:
# Change DataTypes: IsPersonalBest to Bool; TrackStatus to int64
clean_df_laps["IsPersonalBest"] = clean_df_laps["IsPersonalBest"].astype(bool)
clean_df_laps["TrackStatus"] = clean_df_laps["TrackStatus"].astype('int64')
print(clean_df_laps.dtypes)

Time              timedelta64[ns]
Driver                     object
LapTime                   float64
LapNumber                 float64
Stint                     float64
PitOutTime                float64
PitInTime                 float64
Sector1Time               float64
Sector2Time               float64
Sector3Time               float64
IsPersonalBest               bool
Compound                    int64
TyreLife                  float64
FreshTyre                    bool
Team                       object
TrackStatus                 int64
Position                  float64
Deleted                      bool
DeletedReason              object
IsAccurate                   bool
dtype: object


In [12]:
clean_df_laps.isna().sum()

Time                 0
Driver               0
LapTime              0
LapNumber            0
Stint                0
PitOutTime        1168
PitInTime         1168
Sector1Time          0
Sector2Time          0
Sector3Time          0
IsPersonalBest       0
Compound             0
TyreLife             0
FreshTyre            0
Team                 0
TrackStatus          0
Position             0
Deleted              0
DeletedReason        0
IsAccurate           0
dtype: int64

**Sector-Speed Table**<br>
A Subset table of Lap Data containing sectorTimes and speeds for each lap

In [13]:
speed_drop = ['Time','DriverNumber','Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime','TyreLife', 'FreshTyre', 'Team', 'LapStartTime','LapStartDate', 'TrackStatus','DeletedReason','FastF1Generated']
clean_df_speed = sample_df_laps.drop(speed_drop, axis=1)
clean_df_speed.head()

,Driver,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,Position,Deleted,IsAccurate
0,VER,0 days 00:01:17.639000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:31.613000,0 days 00:00:25.440000,292.0,233.0,109.0,297.0,False,MEDIUM,1.0,False,False
1,VER,0 days 00:01:55.479000,2.0,1.0,NaT,0 days 01:04:57.200000,0 days 00:00:31.698000,0 days 00:00:46.293000,0 days 00:00:37.488000,176.0,184.0,NaN,195.0,False,MEDIUM,1.0,False,False
2,VER,0 days 00:02:04.721000,3.0,2.0,0 days 01:05:13.560000,NaT,0 days 00:00:37.877000,0 days 00:00:46.298000,0 days 00:00:40.546000,191.0,61.0,270.0,236.0,False,MEDIUM,1.0,False,False
3,VER,0 days 00:01:09.691000,4.0,2.0,NaT,NaT,0 days 00:00:17.618000,0 days 00:00:30.970000,0 days 00:00:21.103000,293.0,230.0,270.0,297.0,True,MEDIUM,1.0,False,True
4,VER,0 days 00:01:10.026000,5.0,2.0,NaT,NaT,0 days 00:00:17.716000,0 days 00:00:31.158000,0 days 00:00:21.152000,285.0,230.0,270.0,289.0,False,MEDIUM,1.0,False,True


In [14]:
# Setting Driver's full name and encoding the compound tyres
clean_df_speed["Driver"] = clean_df_speed["Driver"].map(driver_map).fillna(clean_df_speed["Driver"])
clean_df_speed["Compound"] = clean_df_speed["Compound"].map(compound_map).fillna(clean_df_speed["Compound"])

# Changing lap and sector times to seconds
speed_sec_col = ["LapTime","Sector1Time","Sector2Time","Sector3Time"]
for t in speed_sec_col:
    clean_df_speed[t] = clean_df_speed[t].dt.total_seconds()

# Clear out pitInTime and pitOutTime
clean_df_speed = clean_df_speed[clean_df_speed["PitInTime"].isna()]
clean_df_speed = clean_df_speed[clean_df_speed["PitOutTime"].isna()]
clean_df_speed.shape

(1229, 18)